# YOLOv8 Traffic Detection Training
This notebook provides a step-by-step guide to training a YOLOv8 model for traffic detection, along with visualizing the training results and confusion matrix.

In [ ]:
# Step 1: Install required libraries
import sys
!{sys.executable} -m pip install ultralytics matplotlib opencv-python Pillow

In [ ]:
# Step 2: Import necessary modules
from ultralytics import YOLO
import os
import glob
from IPython.display import display, Image

## 1. Train the YOLOv8 Model
We will use the `yolov8n.pt` (nano) model as a starting point. It's fast and a good baseline.

In [ ]:
# Step 3: Load a pre-trained YOLOv8 model (nano version)
model = YOLO('yolov8n.pt')

# Start training
# Please adjust epochs, imgsz, and batch size according to your GPU RAM
RESULTS_NAME = 'traffic_training'
results = model.train(
    data='d:/Deteaction_Projects/DETECTION/trafic_data/data_1.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name=RESULTS_NAME
)

## 2. Visualize Training Results
YOLOv8 automatically saves all plots in the `runs/detect/<name>` directory.

In [ ]:
# Define the run directory where the model was trained
run_dir = f'runs/detect/{RESULTS_NAME}'

# Display Training Result Graphs (Losses, mAP, etc.)
results_path = os.path.join(run_dir, 'results.png')
if os.path.exists(results_path):
    print("Training Metrics/Results Graph:")
    display(Image(filename=results_path, width=800))
else:
    print(f"Results graph not found at {results_path}")

## 3. Confusion Matrix
The confusion matrix helps us understand what classes the model is confusing with each other.

In [ ]:
# Display Confusion Matrix
confusion_matrix_path = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(confusion_matrix_path):
    print("Confusion Matrix:")
    display(Image(filename=confusion_matrix_path, width=800))
else:
    print("Confusion matrix not found. Did you run the validation step completely?")

# Display Normalized Confusion Matrix
confusion_matrix_norm_path = os.path.join(run_dir, 'confusion_matrix_normalized.png')
if os.path.exists(confusion_matrix_norm_path):
    print("Normalized Confusion Matrix:")
    display(Image(filename=confusion_matrix_norm_path, width=800))

## 4. Evaluate Model on Validation Set
Let's explicitly evaluate the best model on the validation dataset to get test metrics.

In [ ]:
best_model_path = os.path.join(run_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

# Run validation
metrics = best_model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")

## 5. Visualizing Validation Sample Batches

In [ ]:
val_batch_path = os.path.join(run_dir, 'val_batch0_labels.jpg')
val_pred_path = os.path.join(run_dir, 'val_batch0_pred.jpg')

if os.path.exists(val_batch_path):
    print("Ground Truth Labels for a Validation Batch:")
    display(Image(filename=val_batch_path, width=800))

if os.path.exists(val_pred_path):
    print("\nPredictions for the Validation Batch:")
    display(Image(filename=val_pred_path, width=800))

## 6. Run Inference on a Sample Image

In [ ]:
# Get a random image from the validation dataset to test
val_images = glob.glob('d:/Deteaction_Projects/DETECTION/trafic_data/valid/images/*.jpg')
if val_images:
    sample_img = val_images[0]
    
    # Run prediction
    prediction = best_model.predict(source=sample_img, save=True, conf=0.25)
    
    # Display the result
    # Ultralytics saves predictions in runs/detect/predict[number] by default when save=True
    predict_dirs = glob.glob('runs/detect/predict*')
    latest_predict_dir = max(predict_dirs, key=os.path.getmtime)
    
    predicted_img_path = os.path.join(latest_predict_dir, os.path.basename(sample_img))
    
    if os.path.exists(predicted_img_path):
        print("Sample Prediction:")
        display(Image(filename=predicted_img_path, width=800))
else:
    print("No validation images found. Check the path.")